# Fine-tune informes SI I — Qwen2.5-7B-Instruct (Colab)

**Modelo:** `unsloth/Qwen2.5-7B-Instruct`  
**Método:** QLoRA 4-bit + LoRA (Unsloth)  
**GPU:** Runtime → Cambiar tipo de runtime → **T4** (gratis) o mejor (Pro)

### Antes de empezar
1. Sube `training/colab_bundle.zip` (está en tu repo local) **o** monta Drive con el zip.
2. Ejecuta las celdas en orden.
3. Al final descarga `adapter.zip`.

Settings **T4 16GB (anti-OOM)**: `max_seq_length=1024`, batch=1, grad_accum=16, **sin eval durante train**.


## 1) Comprobar GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Activa GPU: Runtime → Cambiar tipo de runtime → T4"
print("CUDA OK:", torch.cuda.get_device_name(0), "| VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1))

## 2) Instalar Unsloth y dependencias

Reinicia el runtime **solo si** Colab lo pide tras instalar. Luego vuelve a ejecutar desde la celda de GPU.

In [ ]:
%%capture
# Instalación recomendada Unsloth para Colab
!pip install -q --upgrade pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q datasets transformers sentencepiece protobuf
print("Instalación lista")


## 3) Cargar datos

Elige **una** opción y ejecuta solo esa celda.

In [ ]:
# Opción A — Subir colab_bundle.zip desde tu PC
from google.colab import files
import zipfile, pathlib

uploaded = files.upload()  # selecciona training/colab_bundle.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content")
ROOT = pathlib.Path("/content/training")
assert (ROOT / "data/train_messages.jsonl").exists(), "Zip incorrecto: falta data/train_messages.jsonl"
print("Datos en", ROOT)

In [ ]:
# Opción B — Google Drive (pon el zip en MyDrive/pipeeline/colab_bundle.zip)
from google.colab import drive
import zipfile, pathlib

drive.mount("/content/drive")
ZIP = pathlib.Path("/content/drive/MyDrive/pipeeline/colab_bundle.zip")  # ajusta la ruta
assert ZIP.exists(), f"No está el zip en {ZIP}"
with zipfile.ZipFile(ZIP, "r") as z:
    z.extractall("/content")
ROOT = pathlib.Path("/content/training")
print("Datos en", ROOT)

## 4) Configuración (Qwen2.5-7B-Instruct / T4)

In [ ]:
from pathlib import Path
import os

# Si usaste Opción A/B, ROOT ya existe. Si no, descomenta:
# ROOT = Path("/content/training")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

CFG = {
    "model": "unsloth/Qwen2.5-7B-Instruct",
    "max_seq_length": 1024,  # T4: 1024 estable; 1536 si aguanta; 2048 suele OOM en eval
    "batch_size": 1,
    "grad_accum": 16,
    "lr": 2e-4,
    "epochs": 3.0,
    "lora_r": 16,
    "lora_alpha": 32,
    "seed": 42,
    "warmup_steps": 2,
}
TRAIN = ROOT / "data" / "train_messages.jsonl"
VAL = ROOT / "data" / "val_messages.jsonl"
OUT = Path("/content/outputs/qwen25-7b-informes")
OUT.mkdir(parents=True, exist_ok=True)
print(CFG)
print("train lines:", sum(1 for _ in open(TRAIN, encoding="utf-8")))


## 5) Cargar modelo + LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CFG["model"],
    max_seq_length=CFG["max_seq_length"],
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CFG["lora_r"],
    lora_alpha=CFG["lora_alpha"],
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CFG["seed"],
)
print("Modelo listo")

## 6) Dataset

In [ ]:
from datasets import load_dataset

def to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_ds = load_dataset("json", data_files=str(TRAIN), split="train").map(to_text)
eval_ds = load_dataset("json", data_files=str(VAL), split="train").map(to_text)
print(train_ds)
print("--- preview ---")
print(train_ds[0]["text"][:500])

## 7) Entrenar

In [ ]:
from trl import SFTTrainer, SFTConfig
import json
import torch
import gc

# Liberar fragmentación de un intento previo
gc.collect()
torch.cuda.empty_cache()

(OUT / "run_config.json").write_text(json.dumps(CFG, indent=2), encoding="utf-8")

sft_args = SFTConfig(
    output_dir=str(OUT),
    per_device_train_batch_size=CFG["batch_size"],
    gradient_accumulation_steps=CFG["grad_accum"],
    learning_rate=CFG["lr"],
    num_train_epochs=CFG["epochs"],
    logging_steps=1,
    save_strategy="epoch",
    # Eval en T4 + 7B suele provocar OOM; evaluamos al final a mano
    eval_strategy="no",
    warmup_steps=CFG["warmup_steps"],
    lr_scheduler_type="cosine",
    seed=CFG["seed"],
    optim="adamw_8bit",
    dataset_text_field="text",
    max_seq_length=CFG["max_seq_length"],
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    # eval_dataset=eval_ds,  # desactivado en T4
    args=sft_args,
)

trainer.train()

adapter_dir = OUT / "adapter"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print("Adapter guardado en", adapter_dir)


## 8) Probar generación rápida

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

system = (ROOT / "system_prompt.txt").read_text(encoding="utf-8")
user = """Redactar las conclusiones de un proyecto de sistema de información para una biblioteca municipal.

Datos de entrada:
{
  "project_name": "Sistema de Información Biblioteca Municipal Central",
  "objectives_met": ["préstamos", "catálogo", "socios", "reportes"],
  "total_use_cases": 22,
  "total_tables": 18,
  "total_tests": 8
}"""

messages = [
    {"role": "system", "content": system},
    {"role": "user", "content": user},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=700, temperature=0.3, do_sample=True)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

## 9) Descargar adapter + copiar a Drive (opcional)

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/qwen25-7b-informes-adapter", "zip", OUT / "adapter")
print("Zip:", zip_path)
files.download(zip_path)

# Opcional: guardar también en Drive
# drive_dst = Path("/content/drive/MyDrive/pipeeline/outputs/qwen25-7b-informes-adapter.zip")
# drive_dst.parent.mkdir(parents=True, exist_ok=True)
# shutil.copy(zip_path, drive_dst)
# print("Copiado a", drive_dst)

## Si hay OOM (out of memory)

1. **Runtime → Reiniciar sesión** (obligatorio tras un OOM).
2. En CFG usa `max_seq_length: 1024` (ya es el default del notebook).
3. Deja `eval_strategy="no"` (la validación en medio del train era lo que petaba).
4. Si sigue fallando: cambia el modelo a `unsloth/Qwen2.5-3B-Instruct`.
